In [3]:
import rasterio
from rasterio import features
import fiona
from shapely.geometry import shape

# Define file paths
tif_path = '/home/btech1/mask.tif'
shape_path = '/home/btech1/shapefile.shp'
output_tif_path = '/home/btech1/isro/output.tif'

# Open the TIFF file
with rasterio.open(tif_path) as src:
    # Read the raster data and metadata
    data = src.read(1)
    profile = src.profile.copy()

    # Initialize a combined mask (False everywhere initially)
    combined_mask = None

    # Read all polygons from the shapefile
    with fiona.open(shape_path, 'r') as shp:
        for feature in shp:
            polygon = shape(feature['geometry'])
            
            # Create a mask for the current polygon (True inside, False outside)
            mask = features.geometry_mask(
                [polygon],
                out_shape=data.shape,
                transform=src.transform,
                invert=True  # True = inside polygon
            )
            
            # Combine masks (OR operation: True if inside ANY polygon)
            if combined_mask is None:
                combined_mask = mask
            else:
                combined_mask |= mask  # Logical OR to include all polygons

    # Identify pixels inside ANY polygon AND with value 1
    target_pixels = (data == 1) & combined_mask

    # Set identified pixels to 0
    data[target_pixels] = 0

    # Write the modified raster to a new TIFF file
    with rasterio.open(output_tif_path, 'w', **profile) as dst:
        dst.write(data, 1)

print("Operation completed successfully for all polygons!")

Operation completed successfully for all polygons!
